# Transformer Implementation (cross attention)

In this notebook, we will implement the Transformer architecture.\
From: https://nlp.seas.harvard.edu/annotated-transformer/

$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$

In [206]:
import os
import copy
import time
import math

import pandas as pd
import torch
import torch.nn as nn

from torch.optim.lr_scheduler import LambdaLR
import altair as alt
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP

In [207]:
DATA_PATH = "kaggle/input/wmt-2014-english-french/wmt14_translate_fr-en_train.csv"
DEVICE = "mps"

In [208]:
class EncoderDecoder(nn.Module):
    """
    A standard Encoder-Decoder architecture. Base for this and many
    other models.
    """

    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator

    def forward(self, src, tgt, src_mask, tgt_mask):
        "Take in and process masked src and target sequences."
        return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)

    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)

In [209]:
class Generator(nn.Module):
    "Define standard linear + softmax generation step. Last step."

    def __init__(self, d_model, vocab):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab)

    def forward(self, x):
        return log_softmax(self.proj(x), dim=-1)

In [210]:
def clones(module, N):
    "Produce N identical layers."
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [211]:
class Encoder(nn.Module):
    "Core encoder is a stack of N layers"

    def __init__(self, layer, N):
        super().__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.feature_amount)

    def forward(self, x, mask):
        "Pass the input (and mask) through each layer in turn."
        for layer in self.layers:
            x = layer(x, mask)
            # There's one normalization per encoder layer
        return self.norm(x)

In [212]:
class LayerNorm(nn.Module):
    "Construct a layernorm module."

    def __init__(self, feature_amount, eps=1e-6):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(feature_amount))
        self.offset = nn.Parameter(torch.zeros(feature_amount))
        self.eps = eps  # To not divide by zero

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.scale * (x - mean) / (std + self.eps) + self.offset

In [213]:
class ResidualConnection(nn.Module):
    """
    A residual connection followed by a layer norm.
    Note for code simplicity the norm is first as opposed to last.
    """

    def __init__(self, feature_amount, dropout_rate):
        super().__init__()
        self.norm = LayerNorm(feature_amount)
        self.dropout = nn.Dropout(dropout_rate) 
        # Zeros randomly a part of the weights during training

    def forward(self, x, sublayer):
        "Apply residual connection to any sublayer with the same size."
        return x + self.dropout(sublayer(self.norm(x)))

In [214]:
class EncoderLayer(nn.Module):
    "Encoder is made up of self-attn and feed forward"

    def __init__(self, feature_amount, self_attn, feed_forward, dropout_rate):
        super().__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(ResidualConnection(feature_amount, dropout_rate), 2)
        self.feature_amount = feature_amount

    def forward(self, x, mask):
        "Follow Figure 1 (left) for connections."
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
        x = self.sublayer[1](x, self.feed_forward)
        return x

In [215]:
class Decoder(nn.Module):
    "Generic N layer decoder with masking."

    def __init__(self, layer, N):
        super().__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.feature_amount)

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)

In [216]:
class DecoderLayer(nn.Module):
    "Decoder is made of 3 layers: self-attn, src-attn, and feed forward"

    def __init__(self, feature_amount, self_attn, src_attn, feed_forward, dropout_rate):
        super().__init__()
        self.feature_amount = feature_amount
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(ResidualConnection(feature_amount, dropout_rate), 3)

    def forward(self, x, memory, src_mask, tgt_mask):
        m = memory
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
        x = self.sublayer[2](x, self.feed_forward)
        return x

In [217]:
def subsequent_mask(sentence_length):
    "Mask out subsequent positions."

    attn_shape = (1, sentence_length, sentence_length)
    subsequent_mask = torch.triu(torch.ones(attn_shape), diagonal=1)
    # Zeros everything under the first line above the diagonal of the matrix
    mask = (subsequent_mask == 0) # True/False instead of 0/1
    return mask


In [218]:
def attention(query, key, value, mask=None, dropout=None):
    "Compute Scaled Dot Product Attention"
    d_k = query.size(-1)  # Query/key space dimension
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
        
    p_attn = scores.softmax(dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)

    return torch.matmul(p_attn, value), p_attn

In [219]:
class MultiHeadedAttention(nn.Module):
    def __init__(self, head_amount, d_model, dropout_rate=0.1):
        "Take in model size and number of heads."
        super().__init__()
        assert d_model % head_amount == 0
        # We assume d_v always equals d_k
        self.d_k = d_model // head_amount
        self.head_amount = head_amount
        self.linears = clones(nn.Linear(d_model, d_model), 4)
        self.attn = None
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, query, key, value, mask=None):
        if mask is not None:
            # Same mask applied to all h heads.
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)

        # 1) Do all the linear projections in batch from d_model => h x d_k
        query, key, value = [
            lin(x).view(nbatches, -1, self.head_amount, self.d_k).transpose(1, 2)
            for lin, x in zip(self.linears, (query, key, value))
        ]

        # 2) Apply attention on all the projected vectors in batch.
        x, self.attn = attention(
            query, key, value, mask=mask, dropout=self.dropout
        )

        # 3) "Concat" using a view and apply a final linear.
        x = (
            x.transpose(1, 2)
            .contiguous()
            .view(nbatches, -1, self.head_amount * self.d_k)
        )
        
        del query, key, value # In case VRAM is overwhelmed
        return self.linears[-1](x)

In [220]:
class PositionwiseFeedForward(nn.Module):
    "Implements FFN equation."

    def __init__(self, d_model, d_ff, dropout_rate=0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.w_1(x)
        x = x.relu()
        x = self.dropout(x)
        x = self.w_2(x)
        return x

In [221]:
class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super().__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.lut(x) * math.sqrt(self.d_model)

In [222]:
class SimpleLossCompute:
    "A simple loss compute and train function."

    def __init__(self, generator, criterion):
        self.generator = generator
        self.criterion = criterion

    def __call__(self, x, y, norm):
        x = self.generator(x)
        sloss = (
            self.criterion(
                x.contiguous().view(-1, x.size(-1)), y.contiguous().view(-1)
            )
            / norm
        )
        return sloss.data * norm, sloss

In [223]:
class PositionalEncoding(nn.Module):
    "Implement the PE function."

    def __init__(self, d_model, dropout, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Compute the positional encodings once in log space.
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, : x.size(1)].requires_grad_(False)
        x = self.dropout(x)
        return x

In [224]:
def make_model(src_vocab, tgt_vocab, N=6, d_model=512, d_ff=2048, h=8, dropout_rate=0.1):
    "Helper: Construct a model from hyperparameters."
    
    c = copy.deepcopy
    attn = MultiHeadedAttention(h, d_model)
    ff = PositionwiseFeedForward(d_model, d_ff, dropout_rate)
    position = PositionalEncoding(d_model, dropout_rate)

    model = EncoderDecoder(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout_rate), N),
        Decoder(DecoderLayer(d_model, c(attn), c(attn), c(ff), dropout_rate), N),
        nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
        nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
        Generator(d_model, tgt_vocab),
    )

    # This was important from their code.
    # Initialize parameters with Glorot / fan_avg.
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    return model

In [225]:
def greedy_decode(model, src, src_mask, max_len, start_symbol):
    memory = model.encode(src, src_mask)
    ys = torch.zeros(1, 1).fill_(start_symbol).type_as(src.data)
    for i in range(max_len - 1):
        out = model.decode(
            memory, src_mask, ys, subsequent_mask(ys.size(1)).type_as(src.data)
        )
        prob = model.generator(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.data[0]
        ys = torch.cat(
            [ys, torch.zeros(1, 1).type_as(src.data).fill_(next_word)], dim=1
        )
    return ys

In [226]:
class Batch:
    """Object for holding a batch of data with mask during training."""

    def __init__(self, src, tgt=None, pad=2):  # 2 = <blank>
        self.src = src
        self.src_mask = (src != pad).unsqueeze(-2)
        if tgt is not None:
            self.tgt = tgt[:, :-1]
            self.tgt_y = tgt[:, 1:]
            self.tgt_mask = self.make_std_mask(self.tgt, pad)
            self.ntokens = (self.tgt_y != pad).data.sum()

    @staticmethod
    def make_std_mask(tgt, pad):
        "Create a mask to hide padding and future words."
        tgt_mask = (tgt != pad).unsqueeze(-2)
        tgt_mask = tgt_mask & subsequent_mask(tgt.size(-1)).type_as(
            tgt_mask.data
        )
        return tgt_mask

In [227]:
class TrainState:
    """Track number of steps, examples, and tokens processed"""

    step: int = 0  # Steps in the current epoch
    accum_step: int = 0  # Number of gradient accumulation steps
    samples: int = 0  # number of examples used
    tokens: int = 0  # number of tokens processed

In [228]:
def run_epoch(
    data_iter,
    model,
    loss_compute,
    optimizer,
    scheduler,
    mode="train",
    accum_iter=1,
    train_state=TrainState(),):
    """Train a single epoch"""
    
    start = time.time()
    total_tokens = 0
    total_loss = 0
    tokens = 0
    n_accum = 0

    for i, batch in enumerate(data_iter):
        out = model.forward(
            batch.src, batch.tgt, batch.src_mask, batch.tgt_mask
        )
        loss, loss_node = loss_compute(out, batch.tgt_y, batch.ntokens)
        # loss_node = loss_node / accum_iter
        if mode == "train" or mode == "train+log":
            loss_node.backward()
            train_state.step += 1
            train_state.samples += batch.src.shape[0]
            train_state.tokens += batch.ntokens
            if i % accum_iter == 0:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                n_accum += 1
                train_state.accum_step += 1
            scheduler.step()

        total_loss += loss
        total_tokens += batch.ntokens
        tokens += batch.ntokens
        if i % 40 == 1 and (mode == "train" or mode == "train+log"):
            lr = optimizer.param_groups[0]["lr"]
            elapsed = time.time() - start
            print(
                (
                    "Epoch Step: %6d | Accumulation Step: %3d | Loss: %6.2f "
                    + "| Tokens / Sec: %7.1f | Learning Rate: %6.1e"
                )
                % (i, n_accum, loss / batch.ntokens, tokens / elapsed, lr)
            )
            start = time.time()
            tokens = 0
        del loss
        del loss_node
    return total_loss / total_tokens, train_state

In [229]:
def rate(step, model_size, factor, warmup):
    """
    we have to default the step to 1 
    to avoid zero raising to negative power.
    """
    if step == 0:
        step = 1
    rate = factor * (
        model_size ** (-0.5) * min(step ** (-0.5), step * warmup ** (-1.5))
    )
    return rate

In [230]:
class LabelSmoothing(nn.Module):
    """
    Implement label smoothing. Prevent trying to reach P = 1 or 0 
    and overfit.
    """

    def __init__(self, feature_amount, padding_idx, smoothing=0.0):
        super().__init__()
        self.criterion = nn.KLDivLoss(reduction="sum")
        self.padding_idx = padding_idx
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.feature_amount = feature_amount
        self.true_dist = None

    def forward(self, x, target):
        assert x.size(1) == self.feature_amount
        true_dist = x.data.clone()
        true_dist.fill_(self.smoothing / (self.feature_amount - 2))
        true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)
        true_dist[:, self.padding_idx] = 0
        mask = torch.nonzero(target.data == self.padding_idx)
        if mask.dim() > 0:
            true_dist.index_fill_(0, mask.squeeze(), 0.0)
        self.true_dist = true_dist
        return self.criterion(x, true_dist.clone().detach())

# Data Loading Pipeline

In [231]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing
from tokenizers.decoders import BPEDecoder

In [ ]:
def train_tokenizer(data, vocab_size=30000, min_frequency=2):

    tokenizer = Tokenizer(BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(special_tokens=["<s>", "<pad>", "</s>", "<unk>"], 
                         vocab_size=vocab_size, 
                         min_frequency=min_frequency)
    tokenizer.train_from_iterator(data, trainer)
    
    # Post-processing: Add <s> at start and end
    tokenizer.post_processor = TemplateProcessing(
        single="<s> $A </s>",
        pair="<s> $A </s> $B:1 </s>:1",
        special_tokens=[
            ("<s>", tokenizer.token_to_id("<s>")),
            ("</s>", tokenizer.token_to_id("</s>")),
        ],
    )
    tokenizer.decoder = BPEDecoder()
    return tokenizer


In [ ]:

def load_tokenizers(data_path=DATA_PATH):

    df = pd.read_csv(data_path)
    train_src = df['fr'].astype(str).tolist() # Source is French
    train_tgt = df['en'].astype(str).tolist() # Target is English
        
    tokenizer_src = train_tokenizer(train_src, vocab_size=30000)
    tokenizer_tgt = train_tokenizer(train_tgt, vocab_size=30000)

    tokenizer_src.save("tokenizer_src.json")
    tokenizer_tgt.save("tokenizer_tgt.json")

    return tokenizer_src, tokenizer_tgt


# Training

In [234]:
def create_dataloaders(
    device,
    vocab_src,
    vocab_tgt,
    # REMOVED: spacy_fr, spacy_en
    batch_size=12000,
    max_padding=128,
    is_distributed=True,
    data_path=DATA_PATH, # Uses global constant now
    src_lang='fr',
    tgt_lang='en'
):
    tokenizer_src = vocab_src
    tokenizer_tgt = vocab_tgt
    
    pad_id = tokenizer_src.token_to_id("<pad>")
    def collate_fn(batch):
        return collate_batch(
            batch,
            tokenizer_src,
            tokenizer_tgt,
            device,
            max_padding=max_padding,
            pad_id=pad_id,
        )
    print(f'Loading data from {data_path}...')
    train_src, train_tgt = load_data(data_path)
    if not train_src or not train_tgt:
         print('Warning: Training files not found. Returning empty dataloaders.')
         train_iter = []
         valid_iter = []
    else:
         split_idx = int(len(train_src) * 0.95)
         train_iter = list(zip(train_src[:split_idx], train_tgt[:split_idx]))
         valid_iter = list(zip(train_src[split_idx:], train_tgt[split_idx:]))
    train_sampler = (
        DistributedSampler(train_iter) if is_distributed else None
    )
    valid_sampler = (
        DistributedSampler(valid_iter) if is_distributed else None
    )
    train_dataloader = DataLoader(
        train_iter,
        batch_size=batch_size,
        shuffle=(train_sampler is None),
        sampler=train_sampler,
        collate_fn=collate_fn,
    )
    valid_dataloader = DataLoader(
        valid_iter,
        batch_size=batch_size,
        shuffle=(valid_sampler is None),
        sampler=valid_sampler,
        collate_fn=collate_fn,
    )
    return train_dataloader, valid_dataloader

In [235]:
def train_worker(
    gpu,
    ngpus_per_node,
    vocab_src,
    vocab_tgt,
    config,
    is_distributed=False,
):
    print(f"Train worker process using GPU: {gpu} for training", flush=True)
    torch.cuda.set_device(gpu) if torch.cuda.is_available() else None

    pad_idx = vocab_tgt.token_to_id("<pad>")
    d_model = 512
    model = make_model(vocab_src.get_vocab_size(), vocab_tgt.get_vocab_size(), N=6)
    
    # Device Selection using global constant if available, else local check
    device = torch.device(DEVICE if 'DEVICE' in globals() else 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
    print(f"Using device: {device}")
    model.to(device)
    
    module = model
    is_main_process = True
    if is_distributed:
        dist.init_process_group(
            "nccl", init_method="env://", rank=gpu, world_size=ngpus_per_node
        )
        model = DDP(model, device_ids=[gpu])
        module = model.module
        is_main_process = (gpu == 0)

    criterion = LabelSmoothing(
        feature_amount=vocab_tgt.get_vocab_size(), padding_idx=pad_idx, smoothing=0.1
    )
    criterion.to(device)

    train_dataloader, valid_dataloader = create_dataloaders(
        device,
        vocab_src,
        vocab_tgt,
        batch_size=config["batch_size"] // ngpus_per_node,
        max_padding=config["max_padding"],
        is_distributed=is_distributed,
        data_path=DATA_PATH
    )

    optimizer = torch.optim.Adam(
        model.parameters(), lr=config["base_lr"], betas=(0.9, 0.98), eps=1e-9
    )
    lr_scheduler = LambdaLR(
        optimizer,
        lr_lambda=lambda step: rate(
            step, d_model, factor=1, warmup=config["warmup"]
        ),
    )
    train_state = TrainState()

    for epoch in range(config["num_epochs"]):
        if is_distributed:
            train_dataloader.sampler.set_epoch(epoch)
            valid_dataloader.sampler.set_epoch(epoch)

        model.train()
        print(f"[GPU{gpu}] Epoch {epoch} Training ====", flush=True)
        _, train_state = run_epoch(
            (Batch(b[0], b[1], pad_idx) for b in train_dataloader),
            model,
            SimpleLossCompute(module.generator, criterion),
            optimizer,
            lr_scheduler,
            mode="train+log",
            accum_iter=config["accum_iter"],
            train_state=train_state,
        )

        if is_main_process:
            file_path = "%s%.2d.pt" % (config["file_prefix"], epoch)
            torch.save(module.state_dict(), file_path)
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

        print(f"[GPU{gpu}] Epoch {epoch} Validation ====", flush=True)
        model.eval()
        sloss = run_epoch(
            (Batch(b[0], b[1], pad_idx) for b in valid_dataloader),
            model,
            SimpleLossCompute(module.generator, criterion),
            DummyOptimizer(),
            DummyScheduler(),
            mode="eval",
        )
        print(sloss)
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    if is_main_process:
        file_path = "%sfinal.pt" % config["file_prefix"]
        torch.save(module.state_dict(), file_path)


In [236]:
def train_distributed_model(vocab_src, vocab_tgt, config):

    ngpus = torch.cuda.device_count()
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "12356"
    print(f"Number of GPUs detected: {ngpus}")
    print("Spawning training processes ...")
    mp.spawn(
        train_worker,
        nprocs=ngpus,
        args=(ngpus, vocab_src, vocab_tgt, config, True),
    )


def train_model(vocab_src, vocab_tgt, config):
    if config["distributed"]:
        train_distributed_model(
            vocab_src, vocab_tgt, config
        )
    else:
        train_worker(
            0, 1, vocab_src, vocab_tgt, config, False
        )


def load_trained_model():
    config = {
        "batch_size": 32,
        "distributed": False,
        "num_epochs": 8,
        "accum_iter": 10,
        "base_lr": 1.0,
        "max_padding": 72,
        "warmup": 3000,
        "file_prefix": "multi30k_model_",
    }
    model_path = "multi30k_model_final.pt"
    if not exists(model_path):
        # Assuming global tokenizer variables exist if run in notebook
        if 'tokenizer_src' in globals() and 'tokenizer_tgt' in globals():
            train_model(tokenizer_src, tokenizer_tgt, config)
        else:
            print("Tokenizers not loaded. Please run training first.")

    if 'tokenizer_src' in globals() and 'tokenizer_tgt' in globals():
        model = make_model(tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size(), N=6)
        model.load_state_dict(torch.load("multi30k_model_final.pt"))
        return model
    return None


In [237]:
def check_outputs(
    valid_dataloader,
    model,
    vocab_src,
    vocab_tgt,
    n_examples=15,
    pad_idx=2,
    eos_string="</s>",
):
    results = [()] * n_examples
    for idx in range(n_examples):
        print("\nExample %d ========\n" % idx)
        try:
            b = next(iter(valid_dataloader))
        except StopIteration:
            print('No more examples in test set.')
            break
        rb = Batch(b[0], b[1], pad_idx)
        
        src_tokens = [
            vocab_src.get_itos()[x] for x in rb.src[0] if x != pad_idx
        ]
        tgt_tokens = [
            vocab_tgt.get_itos()[x] for x in rb.tgt[0] if x != pad_idx
        ]

        print(
            "Source Text (Input)        : "
            + " ".join(src_tokens).replace("\n", "")
        )
        print(
            "Target Text (Ground Truth) : "
            + " ".join(tgt_tokens).replace("\n", "")
        )
        model_out = greedy_decode(model, rb.src, rb.src_mask, 72, 0)[0]
        model_txt = (
            " ".join(
                [vocab_tgt.get_itos()[x] for x in model_out if x != pad_idx]
            ).split(eos_string, 1)[0]
            + eos_string
        )
        print("Model Output               : " + model_txt.replace("\n", ""))
        results[idx] = (rb, src_tokens, tgt_tokens, model_out, model_txt)
    return results


In [238]:
def mtx2df(m, max_row, max_col, row_tokens, col_tokens):
    "convert a dense matrix to a data frame with row and column indices"
    return pd.DataFrame(
        [
            (
                r,
                c,
                float(m[r, c]),
                "%.3d %s"
                % (r, row_tokens[r] if len(row_tokens) > r else "<blank>"),
                "%.3d %s"
                % (c, col_tokens[c] if len(col_tokens) > c else "<blank>"),
            )
            for r in range(m.shape[0])
            for c in range(m.shape[1])
            if r < max_row and c < max_col
        ],
        columns=["row", "column", "value", "row_token", "col_token"],
    )

def attn_map(attn, layer, head, row_tokens, col_tokens, max_dim=30):
    df = mtx2df(
        attn[0, head].data,
        max_dim,
        max_dim,
        row_tokens,
        col_tokens,
    )
    return (
        alt.Chart(data=df)
        .mark_rect()
        .encode(
            x=alt.X("col_token", axis=alt.Axis(title="")),
            y=alt.Y("row_token", axis=alt.Axis(title="")),
            color="value",
            tooltip=["row", "column", "value", "row_token", "col_token"],
        )
        .properties(height=400, width=400)
        .interactive()
    )

def get_encoder(model, layer):
    return model.encoder.layers[layer].self_attn.attn

def get_decoder_self(model, layer):
    return model.decoder.layers[layer].self_attn.attn

def get_decoder_src(model, layer):
    return model.decoder.layers[layer].src_attn.attn

def visualize_layer(model, layer, getter_fn, ntokens, row_tokens, col_tokens):
    attn = getter_fn(model, layer)
    n_heads = attn.shape[1]
    charts = [
        attn_map(
            attn,
            0,
            h,
            row_tokens=row_tokens,
            col_tokens=col_tokens,
            max_dim=ntokens,
        )
        for h in range(n_heads)
    ]
    return alt.vconcat(
        charts[0]
        | charts[2]
        | charts[4]
        | charts[6]
    ).properties(title="Layer %d" % (layer + 1))

In [239]:
def is_interactive_notebook():
    return __name__ == "__main__"


def show_example(fn, args=[]):
    if __name__ == "__main__" and RUN_EXAMPLES:
        return fn(*args)


def execute_example(fn, args=[]):
    if __name__ == "__main__" and RUN_EXAMPLES:
        fn(*args)


class DummyOptimizer(torch.optim.Optimizer):
    def __init__(self):
        self.param_groups = [{"lr": 0}]
        None

    def step(self):
        None

    def zero_grad(self, set_to_none=False):
        None


class DummyScheduler:
    def step(self):
        None

In [240]:
if __name__ == "__main__":
    print("Configuration loaded.")
    
    if os.path.exists(DATA_PATH):
        print(f"Dataset found at {DATA_PATH}. Initializing training...")
        
        # 1. Load Tokenizers
        tokenizer_src, tokenizer_tgt = load_tokenizers(data_dir=DATA_PATH)
        
        # 2. Configure and Train
        config = {
            "batch_size": 32,
            "distributed": False,
            "num_epochs": 10,
            "accum_iter": 10,
            "base_lr": 1.0,
            "max_padding": 72,
            "warmup": 3000,
            "file_prefix": "wmt14_model_",
        }
        
        train_model(tokenizer_src, tokenizer_tgt, config)
    else:
        print("Dataset not found at default Kaggle path. Training skipped.")
        print("To run manually: set 'DATA_PATH' and call train_model(...)")

Configuration loaded.
Dataset found at kaggle/input/wmt-2014-english-french/wmt14_translate_fr-en_train.csv. Initializing training...






Train worker process using GPU: 0 for training
Using device: mps
Loading data from kaggle/input/wmt-2014-english-french/wmt14_translate_fr-en_train.csv...
[GPU0] Epoch 0 Training ====


TypeError: collate_batch() missing 2 required positional arguments: 'tgt_vocab' and 'device'